A notebook to copy some of the data from the catalogue
Authored by Ken Hirata (@kenhira)

### Function imports

In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cf
import uxarray as ux
import intake
import healpix as hp
import holoviews as hv

import easygems.healpix as egh
import easygems.remap as egr

import easygems.healpix as eghp

import cmocean
import geoviews.feature as gf

### (Optional) Using Dask to parallize tasks

In [2]:
import dask 
from dask_jobqueue import PBSCluster
from dask.distributed import Client
from dask.distributed import performance_report

rda_scratch = '/glade/derecho/scratch/khirata/'

cluster = PBSCluster(
    job_name = 'dask-wk24-hpc',
    cores = 16,
    memory = '512GiB',
    local_directory = rda_scratch+'/dask/spill',
    log_directory = rda_scratch + '/dask/logs/',
    resource_spec = 'select=1:ncpus=16:mem=512GB',
    queue = 'casper',
    walltime = '02:00:00',
    #interface = 'ib0'
    interface = 'ext'
)
# cluster = PBSCluster(
#     job_name = 'dask-wk24-hpc',
#     cores = 32,
#     memory = '1024GiB',
#     local_directory = rda_scratch+'/dask/spill',
#     log_directory = rda_scratch + '/dask/logs/',
#     resource_spec = 'select=1:ncpus=32:mem=1024GB',
#     queue = 'casper',
#     walltime = '12:00:00',
#     #interface = 'ib0'
#     interface = 'ext'
# )

cluster.scale(2) # large number may result in an error
# cluster.adapt(minimum=2, maximum=32)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/khirata/hackathon-casper/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/khirata/hackathon-casper/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.177:44225,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/khirata/hackathon-casper/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


### Read in data into UxArray

In [32]:
### From the local file
# uxds_imerg = ux.UxDataset.from_healpix('/glade/derecho/scratch/andrew/hackathon/IMERG_V07B_hp9.zarr')
# native_zoom = 9

native_zoom = 5
uxds_imerg = ux.UxDataset.from_healpix('/glade/derecho/scratch/khirata/imerg_zarr/imerg_precipitation_2018_2023_zmlv%d.zarr' % native_zoom)

### From the catalog
# node_id = 'NCAR'
# cat = intake.open_catalog("https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml")[node_id]
# ds_imerg = cat['IR_IMERG'](zoom=9).to_dask()
# uxds_imerg = ux.UxDataset.from_healpix(ds_imerg)
# native_zoom = 9

### Sepcify time period, variable to store, and target zoom level

In [33]:
st_date = "2018-01-01"
# st_date = "2019-01-01"
# en_date = "2022-01-01"
en_date = "2023-01-01"
time_slice = slice(st_date, en_date)

varname = 'precipitation'

zoom_lev = 4

assert zoom_lev <= native_zoom, "zoom_lev must be less than or equal to native_zoom"

In [34]:
%%time
# uxda_pr_imerg_pre = uxds_imerg[varname].sel(time=time_slice)

### Coarsen down to the specified zoom level
uxda_pr_imerg_fine = uxds_imerg[varname].sel(time=time_slice)
uxda_pr_imerg_fine = uxda_pr_imerg_fine.chunk({'time': 1, 'n_face': uxda_pr_imerg_fine.sizes['n_face']})

# level_down = 9 - zoom_lev
level_down = native_zoom - zoom_lev
uxda_pr_imerg_tmp = uxda_pr_imerg_fine
for i in range(level_down):
    print(f'coarsening level {i}...')
    uxda_pr_imerg_tmp = uxda_pr_imerg_tmp.coarsen(n_face=4).mean()
    uxda_pr_imerg_tmp['crs'].attrs['healpix_nside'] = 2**int(native_zoom - i - 1)
uxda_pr_imerg_pre = uxda_pr_imerg_tmp


coarsening level 0...
CPU times: user 338 ms, sys: 3.84 ms, total: 342 ms
Wall time: 340 ms


In [35]:
rda_scratch = '/glade/derecho/scratch/khirata/'
dirname = 'imerg_zarr'
idname = 'imerg_%s_%d_%d_zmlv%d' % (varname, int(st_date.split('-')[0]), int(en_date.split('-')[0]), zoom_lev)

In [37]:
%%time
import zarr
import numcodecs
import numpy as np

# nch_t = 1024
# nch_c = 1024
# nch_t = 1
# nch_c = uxda_pr_imerg_pre.sizes['n_face']
nch_t = 16 * 4**(9 - zoom_lev)
nch_c = uxda_pr_imerg_pre.sizes['n_face']

def get_dtype(da):
    if np.issubdtype(da.dtype, np.floating):
        return "float32"
    else:
        return da.dtype
        
def get_chunks(dimensions):
    chunks = {
        "time": nch_t,
        "cell": nch_c,
    }

    return tuple((chunks[d] for d in dimensions))

def get_compressor():
    return numcodecs.Blosc("zstd", shuffle=2)



def get_encoding(dataset):
    return {
        var: {
            "compressor": get_compressor(),
            "dtype": get_dtype(dataset[var]),
            "chunks": get_chunks(dataset[var].dims),
        }
        for var in dataset.variables
        if var not in dataset.dims
    }

# uxda_pr_imerg.chunk({'time': 1024, 'n_face': uxda_pr_imerg.sizes['n_face']}).persist()
store = zarr.storage.DirectoryStore(rda_scratch + '%s/%s.zarr' % (dirname, idname), dimension_separator='/')

uxds_pr_imerg = uxda_pr_imerg_pre.swap_dims({'n_face': 'cell'}).to_dataset()

# uxds_pr_imerg.to_zarr(store, encoding=get_encoding(uxds_pr_imerg))
uxds_pr_imerg_rechunked = uxds_pr_imerg.chunk({'time': nch_t, 'cell': nch_c}) #.compute()
uxds_pr_imerg_rechunked.to_zarr(store, encoding=get_encoding(uxds_pr_imerg))


<timed exec>:49: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs


CPU times: user 2min 23s, sys: 7.28 s, total: 2min 30s
Wall time: 12min 54s


In [38]:
uxds_pr_imerg_rechunked

<xarray.Dataset> Size: 647MB
Dimensions:        (cell: 3072, time: 52609)
Coordinates:
  * cell           (cell) float64 25kB 512.0 1.536e+03 ... 3.144e+06 3.145e+06
    crs            float32 4B ...
  * time           (time) datetime64[ns] 421kB 2019-01-01 ... 2022-01-01
Data variables:
    precipitation  (time, cell) float32 646MB dask.array<chunksize=(16384, 3072), meta=np.ndarray>

In [9]:
uxds_pr_imerg_rechunked.chunksizes

Frozen({'time': (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [39]:
cluster.close()
